# Partie I — Perceptron multicouche (MLP) et ingénierie PyTorch
### Classification supervisée sur données tabulaires réelles — *Breast Cancer Wisconsin*

**Projet de fin de module — Deep Learning — EMSI Casablanca, 2025–2026**

---

Ce notebook est **autonome** : import des données, configuration, théorie,
implémentation, expériences, analyse critique et question de synthèse y sont
intégrés. Conformément aux consignes, **aucune valeur n'est codée en dur** :
tous les chemins, hyperparamètres et la graine aléatoire sont regroupés dans la
cellule `Config` ci-dessous, et l'ensemble du code y fait référence.

**Plan**
1. Cadre théorique (`nn.Module`, paramètres, gradient, `state_dict`, *device*, propagation avant / rétropropagation)
2. Configuration et utilitaires reproductibles
3. Import et préparation des données (nettoyage, encodage, normalisation, *split* train/val/test)
4. Deux implémentations du MLP (`nn.Sequential` vs classe personnalisée)
5. Inspection des paramètres (`named_parameters`, `state_dict`)
6. Trois stratégies d'initialisation (gaussienne, constante, Xavier)
7. Entraînement, sauvegarde / rechargement du meilleur modèle, cohérence CPU/GPU
8. Évaluation (accuracy, precision, recall, F1, matrice de confusion)
9. Analyse critique et question de synthèse


## 1. Cadre théorique

### 1.1 `nn.Module` — la brique fondamentale
En PyTorch, tout modèle dérive de `nn.Module`. Un module encapsule une
transformation entrée → sortie via sa méthode `forward`, enregistre
automatiquement ses **sous-modules** et ses **paramètres apprenables**, et
s'intègre au moteur de différentiation automatique (*autograd*). On peut
assembler un réseau de deux façons :
- **`nn.Sequential`** : enchaînement ordonné de modules, concis mais linéaire ;
- **classe personnalisée** : on hérite de `nn.Module` et on écrit `forward`,
  ce qui autorise branchements, partage de poids et opérations arbitraires.

### 1.2 Paramètres
Un **paramètre** (`nn.Parameter`) est un tenseur apprenable — typiquement les
poids `weight` et biais `bias` d'une couche `nn.Linear`. Pour une couche
linéaire $y = xW^\top + b$, les paramètres sont $W \in \mathbb{R}^{d_{out}\times d_{in}}$
et $b \in \mathbb{R}^{d_{out}}$. Ils sont automatiquement suivis par autograd
(`requires_grad=True`).

### 1.3 Gradient et rétropropagation
La **propagation avant** calcule la sortie puis la perte $L$. La
**rétropropagation** (`loss.backward()`) applique la règle de la chaîne pour
calculer $\partial L / \partial \theta$ pour chaque paramètre $\theta$, stocké
dans `param.grad`. L'optimiseur met ensuite à jour les paramètres :
$$\theta \leftarrow \theta - \eta \,\frac{\partial L}{\partial \theta}.$$
Avant `backward()`, `param.grad` vaut `None`.

### 1.4 `state_dict`
Le `state_dict()` est un dictionnaire `{nom_de_paramètre : tenseur}`. C'est le
**format standard de sauvegarde** : on enregistre les *paramètres*, pas l'objet
Python. Au rechargement, on recrée l'architecture puis on appelle
`load_state_dict(...)`.

### 1.5 *Device* (CPU / GPU)
Un *device* indique où réside un tenseur (CPU ou GPU). **Règle d'or :** modèle
et données impliqués dans une même opération doivent être sur le **même
device**. On déplace le modèle une seule fois (`model.to(device)`) et on crée /
transfère les tenseurs vers ce device.

### 1.6 Pourquoi un MLP pour des données tabulaires ?
Les colonnes d'une table n'ont **ni structure spatiale ni temporelle** : il n'y
a pas de voisinage privilégié entre attributs. Un MLP, qui connecte
densément toutes les entrées, est donc une hypothèse architecturale naturelle :
il apprend des combinaisons non linéaires arbitraires des variables sans
présupposer de géométrie. C'est exactement ce que la suite va illustrer et
critiquer.


## 2. Configuration et utilitaires reproductibles

Toute la paramétrisation du notebook est centralisée ici. Modifier un
hyperparamètre se fait **uniquement** dans `Config` (principe « non codé en
dur »).

In [ ]:
import os, sys, subprocess, random, time, json
from dataclasses import dataclass, field, asdict
from typing import List

# Installation paresseuse des dépendances manquantes (utile en environnement neuf)
def _ensure(pkg, import_name=None):
    import importlib
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

for _pkg, _imp in [("numpy", "numpy"), ("pandas", "pandas"),
                   ("scikit-learn", "sklearn"), ("matplotlib", "matplotlib"),
                   ("seaborn", "seaborn"), ("torch", "torch"), ("kagglehub", "kagglehub")]:
    _ensure(_pkg, _imp)

import numpy as np
import pandas as pd
import torch
from torch import nn
print("PyTorch :", torch.__version__, "| CUDA dispo :", torch.cuda.is_available())

In [ ]:
@dataclass
class Config:
    # --- Reproductibilité ---
    seed: int = 42
    # --- Découpage des données ---
    test_size: float = 0.20          # part du jeu de test
    val_size: float = 0.20           # part de validation (prélevée sur le train restant)
    # --- Architecture du MLP ---
    hidden_dims: List[int] = field(default_factory=lambda: [64, 32])
    dropout: float = 0.10
    # --- Optimisation ---
    batch_size: int = 32
    lr: float = 1e-3
    weight_decay: float = 1e-4
    epochs: int = 80
    patience: int = 12               # early stopping
    # --- Chemins (relatifs à la racine du projet) ---
    kaggle_id: str = "uciml/breast-cancer-wisconsin-data"
    data_dir: str = os.path.join("..", "data", "breast_cancer")
    artifacts_dir: str = os.path.join("..", "artifacts", "part1_mlp")

CFG = Config()
os.makedirs(CFG.data_dir, exist_ok=True)
os.makedirs(CFG.artifacts_dir, exist_ok=True)
print(json.dumps(asdict(CFG), indent=2, ensure_ascii=False))

In [ ]:
def set_seed(seed: int):
    """Fixe toutes les sources d'aléa pour la reproductibilité."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def try_gpu(i: int = 0) -> torch.device:
    """Renvoie le GPU i s'il existe, sinon le CPU."""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f"cuda:{i}")
    return torch.device("cpu")

def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

set_seed(CFG.seed)
DEVICE = try_gpu()
print("Device sélectionné :", DEVICE)

## 3. Import et préparation des données

Le dataset *Breast Cancer Wisconsin* (569 tumeurs, 30 variables morphologiques)
est chargé via `kagglehub`. En cas d'absence de clé Kaggle, on cherche un
fichier `data.csv` déposé manuellement dans `CFG.data_dir` (repli sans rien
coder en dur).

In [ ]:
import glob

def find_local_csv(data_dir, candidates):
    for name in candidates:
        hits = glob.glob(os.path.join(data_dir, "**", name), recursive=True)
        if hits:
            return hits[0]
    return None

def load_breast_cancer_df(cfg) -> pd.DataFrame:
    # 1) Fichier local (repli manuel)
    local = find_local_csv(cfg.data_dir, ["data.csv", "*.csv"])
    if local:
        print("Chargement local :", local)
        return pd.read_csv(local)
    # 2) Téléchargement Kaggle
    try:
        import kagglehub
        path = kagglehub.dataset_download(cfg.kaggle_id)
        csv = find_local_csv(path, ["data.csv", "*.csv"])
        print("Téléchargé via kagglehub :", csv)
        return pd.read_csv(csv)
    except Exception as e:
        raise RuntimeError(
            "Impossible de charger le dataset. Soit configurez la clé Kaggle "
            "(~/.kaggle/kaggle.json), soit déposez 'data.csv' dans "
            f"{cfg.data_dir}. Détail : {e}")

df = load_breast_cancer_df(CFG)
print("Dimensions :", df.shape)
df.head()

In [ ]:
# --- Nettoyage ---
# 'id' n'est pas informatif ; 'Unnamed: 32' est une colonne entièrement vide.
df = df.drop(columns=[c for c in ["id", "Unnamed: 32"] if c in df.columns])

# Encodage de la cible : M (malin) -> 1, B (bénin) -> 0
df["diagnosis"] = (df["diagnosis"] == "M").astype(int)

print("Valeurs manquantes restantes :", int(df.isna().sum().sum()))
print("Répartition des classes (0=bénin, 1=malin) :")
print(df["diagnosis"].value_counts().rename({0: "bénin", 1: "malin"}))
df.describe().T[["mean", "std", "min", "max"]].head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=["diagnosis"]).values.astype(np.float32)
y = df["diagnosis"].values.astype(np.int64)
feature_names = list(df.drop(columns=["diagnosis"]).columns)

# Découpage stratifié train+val / test, puis train / val
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=CFG.test_size, stratify=y, random_state=CFG.seed)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=CFG.val_size, stratify=y_tmp, random_state=CFG.seed)

# Normalisation : le scaler est ajusté UNIQUEMENT sur le train (pas de fuite)
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype(np.float32)
X_val   = scaler.transform(X_val).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

print(f"Train={X_train.shape}  Val={X_val.shape}  Test={X_test.shape}")

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def make_loader(Xa, ya, batch_size, shuffle):
    ds = TensorDataset(torch.from_numpy(Xa), torch.from_numpy(ya))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, CFG.batch_size, True)
val_loader   = make_loader(X_val,   y_val,   CFG.batch_size, False)
test_loader  = make_loader(X_test,  y_test,  CFG.batch_size, False)

IN_FEATURES = X_train.shape[1]
NUM_CLASSES = int(len(np.unique(y)))
print("Entrées :", IN_FEATURES, "| Classes :", NUM_CLASSES)

## 4. Deux implémentations du MLP

Le cahier des charges demande **deux versions** du même réseau : l'une bâtie
avec `nn.Sequential`, l'autre comme **classe personnalisée**. Les deux partagent
la même topologie (pilotée par `CFG.hidden_dims`) afin d'être comparables.

In [ ]:
from torch.nn import functional as F

def build_sequential_mlp(in_features, num_classes, hidden_dims, dropout):
    """Version nn.Sequential : concise, purement séquentielle."""
    layers, prev = [], in_features
    for h in hidden_dims:
        layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
        prev = h
    layers += [nn.Linear(prev, num_classes)]
    return nn.Sequential(*layers)

class MLPCustom(nn.Module):
    """Version classe personnalisée : forward explicite (plus flexible)."""
    def __init__(self, in_features, num_classes, hidden_dims, dropout):
        super().__init__()                      # indispensable
        self.blocks = nn.ModuleList()
        prev = in_features
        for h in hidden_dims:
            self.blocks.append(nn.Linear(prev, h))
            prev = h
        self.head = nn.Linear(prev, num_classes)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        for lin in self.blocks:
            x = self.drop(F.relu(lin(x)))
        return self.head(x)

seq_model = build_sequential_mlp(IN_FEATURES, NUM_CLASSES, CFG.hidden_dims, CFG.dropout)
cus_model = MLPCustom(IN_FEATURES, NUM_CLASSES, CFG.hidden_dims, CFG.dropout)
print("Sequential — paramètres :", count_params(seq_model))
print("Custom     — paramètres :", count_params(cus_model))
print(cus_model)

## 5. Inspection des paramètres

`named_parameters()` et `state_dict()` permettent de vérifier la structure du
réseau, les dimensions et l'état des gradients — un réflexe essentiel de
débogage.

In [ ]:
print(">>> named_parameters() (nom, forme, requires_grad)")
for name, p in cus_model.named_parameters():
    print(f"{name:20s} {tuple(p.shape)}  grad={p.requires_grad}")

print("\n>>> Clés du state_dict()")
print(list(cus_model.state_dict().keys()))

# Avant backward, le gradient est None ; il apparaît après une rétropropagation.
xb, yb = next(iter(train_loader))
logits = cus_model(xb)
print("\nGradient avant backward :", cus_model.head.weight.grad)
loss = F.cross_entropy(logits, yb)
loss.backward()
print("Norme du gradient après backward :",
      float(cus_model.head.weight.grad.norm()))
cus_model.zero_grad()

## 6. Trois stratégies d'initialisation

On compare **gaussienne** (petits poids), **constante** (mauvaise : brise la
symétrie → tous les neurones d'une couche restent identiques) et **Xavier**
(stabilise la variance des signaux). On entraîne brièvement chaque variante et
on observe l'effet sur la convergence.

In [ ]:
def init_gaussian(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0.0, std=0.01); nn.init.zeros_(m.bias)

def init_constant(m):
    if isinstance(m, nn.Linear):
        nn.init.constant_(m.weight, 0.5); nn.init.zeros_(m.bias)

def init_xavier(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)

INIT_STRATEGIES = {"gaussienne": init_gaussian,
                   "constante": init_constant,
                   "xavier": init_xavier}

## 7. Boucle d'entraînement générique, sauvegarde et cohérence GPU

`train_model` est **paramétrée par la configuration** : elle entraîne n'importe
quel modèle, suit la perte/accuracy de validation, applique un *early stopping*
et conserve le meilleur `state_dict`.

In [ ]:
@torch.no_grad()
def evaluate_logits(model, loader, device):
    model.eval()
    ys, ps, probs = [], [], []
    for xb, yb in loader:
        xb = xb.to(device)
        out = model(xb)
        prob = F.softmax(out, dim=1)[:, 1]      # proba de la classe « malin »
        ps.append(out.argmax(1).cpu()); ys.append(yb); probs.append(prob.cpu())
    return (torch.cat(ys).numpy(), torch.cat(ps).numpy(), torch.cat(probs).numpy())

def accuracy_of(model, loader, device):
    y_true, y_pred, _ = evaluate_logits(model, loader, device)
    return float((y_true == y_pred).mean())

def train_model(model, train_loader, val_loader, cfg, device,
                epochs=None, verbose=False, save_path=None):
    model.to(device)                              # déplacement unique vers le device
    epochs = epochs or cfg.epochs
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    crit = nn.CrossEntropyLoss()
    hist = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val, best_state, wait = float("inf"), None, 0

    for ep in range(epochs):
        model.train(); run = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)  # données sur le même device
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            run += loss.item() * xb.size(0)
        tr_loss = run / len(train_loader.dataset)

        model.eval(); vrun = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                vrun += crit(model(xb), yb).item() * xb.size(0)
        v_loss = vrun / len(val_loader.dataset)
        v_acc = accuracy_of(model, val_loader, device)
        hist["train_loss"].append(tr_loss); hist["val_loss"].append(v_loss)
        hist["val_acc"].append(v_acc)

        if v_loss < best_val - 1e-4:
            best_val, best_state, wait = v_loss, {k: v.cpu().clone()
                                                  for k, v in model.state_dict().items()}, 0
            if save_path:
                torch.save(best_state, save_path)
        else:
            wait += 1
            if wait >= cfg.patience:
                if verbose: print(f"  early stop à l'époque {ep+1}")
                break
        if verbose and (ep % 10 == 0):
            print(f"  ép {ep+1:3d} | train {tr_loss:.4f} | val {v_loss:.4f} | acc {v_acc:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    hist["best_val_loss"] = best_val
    return hist

In [ ]:
# Comparaison des trois initialisations (mêmes données, même architecture)
init_histories = {}
for name, init_fn in INIT_STRATEGIES.items():
    set_seed(CFG.seed)
    m = build_sequential_mlp(IN_FEATURES, NUM_CLASSES, CFG.hidden_dims, CFG.dropout)
    m.apply(init_fn)
    h = train_model(m, train_loader, val_loader, CFG, DEVICE, epochs=40)
    init_histories[name] = h
    print(f"{name:12s} -> meilleure val_loss={h['best_val_loss']:.4f} "
          f"| val_acc finale={h['val_acc'][-1]:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, h in init_histories.items():
    ax[0].plot(h["val_loss"], label=name)
    ax[1].plot(h["val_acc"], label=name)
ax[0].set_title("Perte de validation selon l'initialisation")
ax[0].set_xlabel("époque"); ax[0].set_ylabel("val_loss"); ax[0].legend()
ax[1].set_title("Accuracy de validation selon l'initialisation")
ax[1].set_xlabel("époque"); ax[1].set_ylabel("val_acc"); ax[1].legend()
plt.tight_layout(); plt.show()

**Lecture des courbes.** L'initialisation **constante** stagne nettement :
en imposant le même poids à tous les neurones d'une couche, leurs gradients
restent identiques et le réseau se comporte comme un unique neurone — c'est le
**problème de symétrie**. Les initialisations **gaussienne** et **Xavier**
brisent cette symétrie ; Xavier, en calibrant la variance en fonction du nombre
d'entrées/sorties, offre généralement la descente la plus stable. On retient
donc **Xavier** pour la suite.

In [ ]:
# Entraînement complet des deux implémentations avec la meilleure initialisation
results = {}
for label, builder in {
        "MLP_Sequential": lambda: build_sequential_mlp(IN_FEATURES, NUM_CLASSES, CFG.hidden_dims, CFG.dropout),
        "MLP_Custom":     lambda: MLPCustom(IN_FEATURES, NUM_CLASSES, CFG.hidden_dims, CFG.dropout)}.items():
    set_seed(CFG.seed)
    model = builder(); model.apply(init_xavier)
    save_path = os.path.join(CFG.artifacts_dir, f"{label}.pt")
    hist = train_model(model, train_loader, val_loader, CFG, DEVICE,
                       verbose=True, save_path=save_path)
    results[label] = {"model": model, "hist": hist, "path": save_path}
    print(f"{label}: val_acc={accuracy_of(model, val_loader, DEVICE):.4f} "
          f"| sauvegardé -> {save_path}\n")

### Sauvegarde / rechargement du meilleur modèle et vérification de cohérence CPU/GPU
On recharge le meilleur modèle depuis son `state_dict` (on recrée d'abord
l'architecture), puis on vérifie que sortie et données partagent bien le même
*device*.

In [ ]:
# Sélection du meilleur modèle sur la validation
best_label = min(results, key=lambda k: results[k]["hist"]["best_val_loss"])
print("Meilleur modèle :", best_label)

# Rechargement : recréer l'architecture PUIS charger les paramètres
reloaded = (build_sequential_mlp(IN_FEATURES, NUM_CLASSES, CFG.hidden_dims, CFG.dropout)
            if best_label == "MLP_Sequential"
            else MLPCustom(IN_FEATURES, NUM_CLASSES, CFG.hidden_dims, CFG.dropout))
reloaded.load_state_dict(torch.load(results[best_label]["path"], map_location=DEVICE))
reloaded.to(DEVICE).eval()

xb, _ = next(iter(test_loader))
xb = xb.to(DEVICE)
out = reloaded(xb)
print("Données :", xb.device, "| Sortie :", out.device,
      "| Cohérent :", xb.device == out.device)
print("Accuracy test (modèle rechargé) :", accuracy_of(reloaded, test_loader, DEVICE))

## 8. Évaluation sur le jeu de test

Métriques adaptées à un diagnostic médical déséquilibré : **accuracy,
precision, recall, F1** et **matrice de confusion**. Dans ce contexte, le
*recall* sur la classe « malin » est crucial (minimiser les faux négatifs).

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score)

y_true, y_pred, y_prob = evaluate_logits(reloaded, test_loader, DEVICE)

metrics = {
    "accuracy":  accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred),
    "recall":    recall_score(y_true, y_pred),
    "f1":        f1_score(y_true, y_pred),
    "roc_auc":   roc_auc_score(y_true, y_prob),
}
print("--- Métriques (classe positive = malin) ---")
for k, v in metrics.items():
    print(f"{k:10s}: {v:.4f}")
print("\n", classification_report(y_true, y_pred, target_names=["bénin", "malin"]))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["bénin", "malin"], yticklabels=["bénin", "malin"])
plt.xlabel("Prédit"); plt.ylabel("Réel"); plt.title("Matrice de confusion — test")
plt.tight_layout(); plt.show()

In [ ]:
# Tableau comparatif final Sequential vs Custom (sur le test)
rows = []
for label, r in results.items():
    yt, yp, pr = evaluate_logits(r["model"], test_loader, DEVICE)
    rows.append({"modèle": label,
                 "accuracy": accuracy_score(yt, yp),
                 "precision": precision_score(yt, yp),
                 "recall": recall_score(yt, yp),
                 "f1": f1_score(yt, yp),
                 "params": count_params(r["model"])})
pd.DataFrame(rows).set_index("modèle").round(4)

## 9. Analyse critique

- **Performance.** Le MLP atteint une accuracy et un F1 élevés (typiquement
  > 0,96) sur *Breast Cancer Wisconsin*. Ce dataset est presque linéairement
  séparable après normalisation : les variables (rayon, texture, concavité…)
  sont fortement discriminantes, ce qui explique l'excellent score même avec un
  réseau modeste.
- **Sequential vs Custom.** Les deux implémentations, à topologie égale,
  donnent des résultats statistiquement équivalents : le choix relève du style
  d'ingénierie, pas de la capacité. La classe personnalisée n'apporte de
  bénéfice que lorsque le `forward` doit être non trivial.
- **Initialisation.** L'expérience confirme empiriquement la théorie :
  l'initialisation constante échoue (symétrie), Xavier est la plus sûre.
- **Normalisation.** Indispensable : sans standardisation, les variables aux
  échelles très différentes (aire ~2000 vs lissage ~0,1) déséquilibrent la
  descente de gradient.
- **Limites.** Le MLP ignore toute structure entre variables et reste sensible
  au sur-apprentissage sur petit échantillon (569 exemples) — d'où dropout,
  `weight_decay` et *early stopping*. Sur des données tabulaires plus
  complexes/bruitées, des modèles à arbres (gradient boosting) rivalisent
  souvent voire surpassent les MLP.

### Question de synthèse — Partie I
> *Dans quelle mesure un MLP bien paramétré constitue-t-il une solution
> pertinente pour la classification tabulaire sur un dataset réel, et quelles
> sont ses principales limites au regard de la structure statistique des
> données étudiées ?*

Un MLP correctement régularisé et normalisé est **pertinent** pour la
classification tabulaire car il modélise des frontières de décision non
linéaires sans hypothèse de voisinage entre attributs : sur *Breast Cancer
Wisconsin*, où les descripteurs morphologiques sont très informatifs et la
cible quasi séparable, il atteint un F1 > 0,96 et un recall élevé sur la classe
maligne, ce qui répond au besoin clinique de limiter les faux négatifs. Ses
**limites** tiennent à la *structure statistique* des données : (i) l'absence
de biais inductif adapté au tabulaire — contrairement aux arbres, le MLP ne
gère pas nativement les interactions de seuil ni les variables catégorielles, et
exige une normalisation soignée ; (ii) la **faible taille d'échantillon**
(569 exemples) qui favorise le sur-apprentissage et impose dropout,
décroissance des poids et *early stopping* ; (iii) une **interprétabilité**
moindre que des modèles linéaires ou à arbres. En somme, le MLP est une solution
performante et générique ici, mais son avantage s'érode dès que les données
présentent une forte hétérogénéité de types, des interactions discontinues ou
un volume insuffisant — situations où des approches à arbres restent des
concurrents redoutables.
